In [ ]:
import argparse
import sqlite3
import struct
import time
from pathlib import Path

# Database development
This notebook describes the step by step process of the developement of the application database. 

## Setup
The database is an SQLite database, residing on disk. We chose SQLite because of:
- Fits the laptop constraint. The README requires a modest footprint on the professor’s laptop. SQLite is a single file with no server to install or run.
Nothing extra to set up. Python’s built-in sqlite3 module is enough, so teammates can clone the repo and run it.
- Matches your data size. The cache is bounded, so you’ll have thousands to tens of thousands of rows. That is well within what SQLite handles.
- Fast local lookups. There’s no network hop, so an exact cache hit takes well under a millisecond, which helps the sub-second target.
- Easy resets for benchmarks. A cold start is deleting the file, and a warm start is copying one. That makes cold-versus-warm comparisons repeatable.
- Vector search is covered. sqlite-vec provides it, with NumPy over blob columns as a fallback. At your scale, brute-force search takes milliseconds.
- Keyword search is built in. FTS5 can index evidence passages for hybrid retrieval at no extra cost.
- Concurrency is manageable. WAL mode plus a single writer queue fits your bounded-queue design, so readers and the writer don’t block each other.
- Simple to ship. The demo and the benchmark data are one file.
- Portable if you outgrow it. If all database access goes through a small interface, you could move to Postgres + pgvector later without changing the other workstream

The database will follow this schema:
https://dbdiagram.io/d/Fact-check-DB-6ad76e2cf65bbc60f6de9441

### Code